# Chapter 11 explore: Evaluating a Fine-Tuned Domain Model

Interactive companion to `code/chapter_11/eval_finetuned_model.py`. Builds a real, held-out evaluation set from the one report never used in any training set in this book, then scores Chapter 8's fine-tuned checkpoint three ways: exact-match, partial-credit word overlap, and perplexity. Needs Chapter 8's checkpoint to exist first (`python code/chapter_08/finetune_at_scale.py`, ~30 min).

In [1]:
import sys
sys.path.insert(0, "../code/chapter_01")
sys.path.insert(0, "../code/chapter_02")
sys.path.insert(0, "../code/chapter_06")
sys.path.insert(0, "../code/chapter_07")
sys.path.insert(0, "../code/chapter_09")
sys.path.insert(0, "../code/chapter_10")
sys.path.insert(0, "../code/chapter_11")

from load_local_model import MODEL_NAME, load_model_and_tokenizer
from hybrid_rag_finetune import latest_checkpoint
from eval_finetuned_model import build_held_out_eval_set, evaluate, perplexity
from peft import PeftModel

eval_set = build_held_out_eval_set()
print(f"{len(eval_set)} held-out examples, report never used in any training set")

model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
lora_model = PeftModel.from_pretrained(model, latest_checkpoint())
print("Ready.")

8 held-out examples, report never used in any training set


`torch_dtype` is deprecated! Use `dtype` instead!


Ready.


Score every held-out example two ways: exact-match and partial-credit overlap.

In [2]:
results = evaluate(lora_model, tokenizer, eval_set)
for r in results:
    print(f"  exact={r['exact_match']}  overlap={r['overlap_score']:.2f}  generated={r['generated']!r}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  exact=False  overlap=0.00  generated='Production Casing Run Csg & Cement Rig up casing'
  exact=False  overlap=0.17  generated='Production Casing Run Csg & Cement Rig up casing'
  exact=False  overlap=0.17  generated='Production Casing Run Csg & Cement Rig up cement'
  exact=False  overlap=0.17  generated='Production Casing Run Csg & Cement Rig up cement'
  exact=False  overlap=0.17  generated='Production Casing Run Csg & Cement Rig up casing'
  exact=False  overlap=0.17  generated='Production Casing Run Csg & Cement Rig up casing'
  exact=False  overlap=0.33  generated='Production Casing Run Csg & Cement Rig up cement'
  exact=False  overlap=0.17  generated='Production Casing Run Csg & Cement Rig up casing'


Compare perplexity on the same held-out text -- base model vs. fine-tuned.

In [3]:
texts = [e["output"] for e in eval_set]
base_model, _ = load_model_and_tokenizer(MODEL_NAME)
print("Base model perplexity:     ", round(perplexity(base_model, tokenizer, texts), 2))
print("Fine-tuned model perplexity:", round(perplexity(lora_model, tokenizer, texts), 2))

Base model perplexity:      159.91


Fine-tuned model perplexity: 25.03
